In [6]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.utils import Sequence
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

class GeneradorSegmentacionTFG(Sequence):
    def __init__(self, carpeta_parches, lista_archivos, batch_size=64, shuffle=True):
        self.carpeta = carpeta_parches
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.archivos_X = lista_archivos
        self.num_clases = 19
        self.indices = np.arange(len(self.archivos_X))
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __len__(self):
        return int(np.ceil(len(self.archivos_X) / self.batch_size))

    def __getitem__(self, index):
        indices_batch = self.indices[index * self.batch_size:(index + 1) * self.batch_size]
        batch_X = []
        batch_Y = []
        
        for idx in indices_batch:
            nom_X = self.archivos_X[idx]
            nom_Y = nom_X.replace('_X.npy', '_Y.npy')
            
            parche_X = np.load(os.path.join(self.carpeta, nom_X)).astype(np.float32)
            parche_Y = np.load(os.path.join(self.carpeta, nom_Y)).astype(np.int32)
            
            if parche_X.shape[0] == 4:
                parche_X = np.transpose(parche_X, (1, 2, 0))
            
            parche_X = parche_X[:48, :48, :]
            parche_Y = parche_Y[:48, :48]
            
            mask_one_hot = tf.one_hot(parche_Y, depth=self.num_clases, dtype=tf.float32).numpy()
            
            batch_X.append(parche_X)
            batch_Y.append(mask_one_hot)
            
        if len(batch_X) == 0:
            return (np.zeros((1, 48, 48, 4), dtype=np.float32), 
                    np.zeros((1, 48, 48, self.num_clases), dtype=np.float32))
            
        return np.array(batch_X), np.array(batch_Y)

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

def categorical_focal_loss(gamma=2.0, alpha=0.25):
    def focal_loss_fixed(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        cross_entropy = -y_true * tf.math.log(y_pred)
        focal_weight = tf.math.pow(1.0 - y_pred, gamma)
        loss = alpha * focal_weight * cross_entropy
        return tf.reduce_mean(tf.reduce_sum(loss, axis=-1))
    return focal_loss_fixed

def build_unet_with_bigearthnet_encoder(input_shape, num_classes, weights_path=None):
    entradas = layers.Input(shape=input_shape)
    init = 'he_normal'
    reg = tf.keras.regularizers.l2(1e-6)
    
    x = layers.Normalization(axis=-1)(entradas)
    
    c1 = layers.Conv2D(32, (3, 3), padding='same', activation=None, kernel_initializer=init, kernel_regularizer=reg, name='enc_conv1_1')(x)
    c1 = layers.BatchNormalization(name='enc_bn1_1')(c1)
    c1 = layers.Activation('gelu')(c1)
    c1 = layers.Conv2D(32, (3, 3), padding='same', activation=None, kernel_initializer=init, kernel_regularizer=reg, name='enc_conv1_2')(c1)
    c1 = layers.BatchNormalization(name='enc_bn1_2')(c1)
    c1 = layers.Activation('gelu')(c1)
    p1 = layers.MaxPooling2D((2, 2))(c1)
    
    c2 = layers.Conv2D(64, (3, 3), padding='same', activation=None, kernel_initializer=init, kernel_regularizer=reg, name='enc_conv2_1')(p1)
    c2 = layers.BatchNormalization(name='enc_bn2_1')(c2)
    c2 = layers.Activation('gelu')(c2)
    c2 = layers.Conv2D(64, (3, 3), padding='same', activation=None, kernel_initializer=init, kernel_regularizer=reg, name='enc_conv2_2')(c2)
    c2 = layers.BatchNormalization(name='enc_bn2_2')(c2)
    c2 = layers.Activation('gelu')(c2)
    p2 = layers.MaxPooling2D((2, 2))(c2)
    
    bottleneck = layers.Conv2D(128, (3, 3), padding='same', activation=None, kernel_initializer=init, kernel_regularizer=reg, name='enc_bottleneck')(p2)
    bottleneck = layers.BatchNormalization(name='enc_bn_bottle')(bottleneck)
    bottleneck = layers.Activation('gelu')(bottleneck)
    
    if weights_path and os.path.exists(weights_path):
        encoder_temporal = models.Model(inputs=entradas, outputs=bottleneck)
        encoder_temporal.load_weights(weights_path, by_name=True, skip_mismatch=True)
        
    u3 = layers.Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same', kernel_initializer=init, kernel_regularizer=reg)(bottleneck)
    merge3 = layers.concatenate([u3, c2]) 
    c3 = layers.Conv2D(64, (3, 3), padding='same', activation=None, kernel_initializer=init, kernel_regularizer=reg)(merge3)
    c3 = layers.BatchNormalization()(c3)
    c3 = layers.Activation('gelu')(c3)
    
    u4 = layers.Conv2DTranspose(32, (2, 2), strides=(2, 2), padding='same', kernel_initializer=init, kernel_regularizer=reg)(c3)
    merge4 = layers.concatenate([u4, c1]) 
    c4 = layers.Conv2D(32, (3, 3), padding='same', activation=None, kernel_initializer=init, kernel_regularizer=reg)(merge4)
    c4 = layers.BatchNormalization()(c4)
    c4 = layers.Activation('gelu')(c4)
    
    salidas = layers.Conv2D(num_classes, (1, 1), activation='softmax', kernel_initializer=init)(c4)
    return models.Model(inputs=entradas, outputs=salidas)

carpeta_dataset = r"D:\TFG\dataset_parches_50x50"
ruta_pesos_bigearthnet = r"D:\TFG\pesos_bigearthnet_encoder.h5"
ruta_salida_modelo = r"D:\TFG\modelo3.keras"

todos_los_X_total = sorted([f for f in os.listdir(carpeta_dataset) if f.endswith('_X.npy')])

CLASES_MINORITARIAS = [3, 7, 12, 18] 

archivos_con_minoritarias = []
archivos_comunes = []

for f in todos_los_X_total[:150000]:
    nom_Y = f.replace('_X.npy', '_Y.npy')
    ruta_Y = os.path.join(carpeta_dataset, nom_Y)
    parche_Y = np.load(ruta_Y)
    contiene_minoritarias = any(clase in parche_Y for clase in CLASES_MINORITARIAS)
    if contiene_minoritarias:
        archivos_con_minoritarias.append(f)
    else:
        archivos_comunes.append(f)

TAMANO_FINAL_ENTRENAMIENTO = 100000
cupo_restante = TAMANO_FINAL_ENTRENAMIENTO - len(archivos_con_minoritarias)

np.random.seed(4215)
np.random.shuffle(archivos_comunes)

if cupo_restante > 0:
    archivos_prueba = archivos_con_minoritarias + archivos_comunes[:cupo_restante]
else:
    archivos_prueba = archivos_con_minoritarias[:TAMANO_FINAL_ENTRENAMIENTO]

np.random.shuffle(archivos_prueba)

archivos_estudio_final = [f for f in todos_los_X_total if f not in archivos_prueba][:30000]

archivos_train, archivos_val = train_test_split(archivos_prueba, test_size=0.2, random_state=42)

generador_train = GeneradorSegmentacionTFG(carpeta_dataset, archivos_train, batch_size=64, shuffle=True)
generador_val = GeneradorSegmentacionTFG(carpeta_dataset, archivos_val, batch_size=64, shuffle=False)
generador_estudio = GeneradorSegmentacionTFG(carpeta_dataset, archivos_estudio_final, batch_size=64, shuffle=False)

model = build_unet_with_bigearthnet_encoder((48, 48, 4), 19, weights_path=ruta_pesos_bigearthnet)

for layer in model.layers:
    if layer.name.startswith('enc_'):
        layer.trainable = False

model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3, clipvalue=0.5),
    loss=categorical_focal_loss(gamma=2.0, alpha=0.25),
    metrics=[tf.keras.metrics.CategoricalAccuracy(name='accuracy')]
)

model.fit(
    generador_train, 
    validation_data=generador_val, 
    epochs=4, 
    callbacks=[callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=1, verbose=1)]
)

for layer in model.layers:
    layer.trainable = True

model.compile(
    optimizer=optimizers.Adam(learning_rate=2e-4, clipvalue=0.5), 
    loss=categorical_focal_loss(gamma=2.0, alpha=0.25),
    metrics=[tf.keras.metrics.CategoricalAccuracy(name='accuracy')]
)

mis_callbacks = [
    callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    callbacks.ModelCheckpoint(filepath=ruta_salida_modelo, monitor='val_loss', save_best_only=True, save_weights_only=False, verbose=1)
]

history = model.fit(
    generador_train,
    validation_data=generador_val,
    epochs=36,
    callbacks=mis_callbacks
)

model = models.load_model(ruta_salida_modelo, custom_objects={'focal_loss_fixed': categorical_focal_loss(gamma=2.0, alpha=0.25)})

todos_reales = []
todas_predicciones = []

for i in range(len(generador_estudio)):
    x_batch, y_batch = generador_estudio[i]
    preds = model.predict_on_batch(x_batch)
    clase_real = np.argmax(y_batch, axis=-1)
    clase_pred = np.argmax(preds, axis=-1)
    todos_reales.append(clase_real.flatten())
    todas_predicciones.append(clase_pred.flatten())

todos_reales = np.concatenate(todos_reales)
todas_predicciones = np.concatenate(todas_predicciones)

nombres_clases = [f"Clase {i}" for i in range(19)]
print(classification_report(todos_reales, todas_predicciones, target_names=nombres_clases, digits=4))

iou_por_clase = []
for clase_id in range(19):
    interseccion = np.sum((todos_reales == clase_id) & (todas_predicciones == clase_id))
    union = np.sum((todos_reales == clase_id) | (todas_predicciones == clase_id))
    if union == 0:
        iou = float('nan')
    else:
        iou = interseccion / union
        iou_por_clase.append(iou)
    print(f"Clase {clase_id} IoU: {iou:.4f}" if not np.isnan(union) else f"Clase {clase_id} IoU: NaN")

mIoU = np.nanmean(iou_por_clase)
print(f"Mean IoU: {mIoU:.4f}")

Epoch 1/4
1250/1250 [==============================] - 3916s 3s/step - loss: 0.0791 - accuracy: 0.4002 - val_loss: 0.0648 - val_accuracy: 0.4123 - lr: 0.0010
Epoch 2/4
1250/1250 [==============================] - 3848s 3s/step - loss: 0.0616 - accuracy: 0.4181 - val_loss: 0.0611 - val_accuracy: 0.4219 - lr: 0.0010
Epoch 3/4
1250/1250 [==============================] - ETA: 0s - loss: 0.0602 - accuracy: 0.4204
Epoch 3: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
1250/1250 [==============================] - 3429s 3s/step - loss: 0.0602 - accuracy: 0.4204 - val_loss: 0.0641 - val_accuracy: 0.4113 - lr: 0.0010
Epoch 4/4
1250/1250 [==============================] - 1223s 979ms/step - loss: 0.0584 - accuracy: 0.4233 - val_loss: 0.0593 - val_accuracy: 0.4224 - lr: 5.0000e-04
Epoch 1/36
 448/1250 [=========>....................] - ETA: 14:07 - loss: 0.0674 - accuracy: 0.4247

KeyboardInterrupt: 